In [9]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("Chinook_Sqlite.sqlite")

# Create a cursor object
cursor = conn.cursor()


In [10]:
import sqlite3
import os

# --- Database Connection ---
db_file = 'Chinook_Sqlite.sqlite' # Make sure this file is in the same directory or provide the correct path

if not os.path.exists(db_file):
    print(f"Error: Database file '{db_file}' not found.")
    # Exit or handle the error appropriately
    exit()

try:
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    print("Database connected successfully.")
except sqlite3.Error as e:
    print(f"Error connecting to database: {e}")
    exit()

# --- Execution Function ---
def execute_query(query, display_n_rows=5):
    """Execute a SQL query and return the results."""
    #print("\n--- Query ---")
    #print(query)
    #print("\n--- Result ---")
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        if not rows:
            print("No results found.")
        else:
            # Print header (optional, but helpful)
            col_names = [description[0] for description in cursor.description]
            print("=========")
            print(tuple(col_names))
            print("---------")
            # Print rows
            for row in rows[:display_n_rows]:
                print(row)
        print(f"Total rows: {len(rows)}")
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")
    print("="*30)


Database connected successfully.


**Top 5 Customers by Total Spending:**

Find the customers who have spent the most money in total across all their invoices.

In [11]:
query = """
WITH CustomerSpending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) AS TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
)
SELECT
    CustomerId,
    FirstName,
    LastName,
    TotalSpent
FROM CustomerSpending
ORDER BY TotalSpent DESC
LIMIT 5;
"""
execute_query(query)

('CustomerId', 'FirstName', 'LastName', 'TotalSpent')
---------
(6, 'Helena', 'Holý', 49.620000000000005)
(26, 'Richard', 'Cunningham', 47.620000000000005)
(57, 'Luis', 'Rojas', 46.62)
(45, 'Ladislav', 'Kovács', 45.62)
(46, 'Hugh', "O'Reilly", 45.62)
Total rows: 5


In [12]:
query = """
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) AS TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
    
    ORDER BY TotalSpent DESC
    LIMIT 5;
"""
execute_query(query)

('CustomerId', 'FirstName', 'LastName', 'TotalSpent')
---------
(6, 'Helena', 'Holý', 49.620000000000005)
(26, 'Richard', 'Cunningham', 47.620000000000005)
(57, 'Luis', 'Rojas', 46.62)
(45, 'Ladislav', 'Kovács', 45.62)
(46, 'Hugh', "O'Reilly", 45.62)
Total rows: 5


**Sales Agent Performance:**

List each sales support agent and the total sales amount they have generated (based on the customers they support).

In [13]:
query = """
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS AgentName,
    SUM(i.Total) AS TotalSalesGenerated
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
JOIN Invoice i ON c.CustomerId = i.CustomerId
WHERE e.Title = 'Sales Support Agent' 
GROUP BY e.EmployeeId, AgentName
ORDER BY TotalSalesGenerated DESC;
"""
execute_query(query)

('EmployeeId', 'AgentName', 'TotalSalesGenerated')
---------
(3, 'Jane Peacock', 833.0400000000016)
(4, 'Margaret Park', 775.4000000000005)
(5, 'Steve Johnson', 720.1600000000011)
Total rows: 3


**Most Popular Genre per Country:**

For each country, find the music genre that has the most tracks purchased (based on invoice lines). If there's a tie, list all tied genres.

In [14]:
query = """
WITH CountryGenreSales AS (
    SELECT
        c.Country,
        g.Name AS GenreName,
        COUNT(il.InvoiceLineId) AS TracksSold
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.Name
),
RankedGenreSales AS (
    SELECT
        Country,
        GenreName,
        TracksSold,
        RANK() OVER (PARTITION BY Country ORDER BY TracksSold DESC) as RankNum
    FROM CountryGenreSales
)
SELECT
    Country,
    GenreName,
    TracksSold
FROM RankedGenreSales
WHERE RankNum = 1
ORDER BY Country, GenreName;
"""
execute_query(query)

('Country', 'GenreName', 'TracksSold')
---------
('Argentina', 'Alternative & Punk', 9)
('Argentina', 'Rock', 9)
('Australia', 'Rock', 22)
('Austria', 'Rock', 15)
('Belgium', 'Rock', 21)
Total rows: 25


In [15]:
query = """
WITH CountryGenreSales AS (
    SELECT
        c.Country,
        g.Name AS GenreName,
        COUNT(il.InvoiceLineId) AS TracksSold
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.Name
),
SELECT
    Country,
    GenreName,
    TracksSold,
    RANK() OVER (PARTITION BY Country ORDER BY TracksSold DESC) as RankNum
FROM CountryGenreSales
WHERE RankNum = 1
ORDER BY Country, GenreName;
"""
execute_query(query)

Error executing query: near "SELECT": syntax error


In [16]:
**4. Tracks Never Purchased:**
Find all tracks that have never appeared on any invoice line.

SyntaxError: invalid syntax (1586484549.py, line 1)

In [ ]:
SELECT
    t.TrackId,
    t.Name AS TrackName,
    a.Title AS AlbumTitle,
    ar.Name AS ArtistName
FROM Track t
JOIN Album a ON t.AlbumId = a.AlbumId
JOIN Artist ar ON a.ArtistId = ar.ArtistId
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE il.InvoiceLineId IS NULL -- The core check for non-existence in InvoiceLine
ORDER BY ArtistName, AlbumTitle, TrackName;

In [ ]:
** 5. Customer Spending vs. Average:**
List customers who have spent more than the average total spending across all customers.

In [ ]:
WITH CustomerTotalSpending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) as TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
),
AverageSpending AS (
    SELECT AVG(TotalSpent) as AvgSpent FROM CustomerTotalSpending
)
SELECT
    cts.CustomerId,
    cts.FirstName,
    cts.LastName,
    cts.TotalSpent
FROM CustomerTotalSpending cts
CROSS JOIN AverageSpending avs -- Use CROSS JOIN to get the average available for comparison
WHERE cts.TotalSpent > avs.AvgSpent
ORDER BY cts.TotalSpent DESC;

In [ ]:
**6. Artist with Most Genres:**
Find the artist who has tracks covering the widest variety of genres.


In [ ]:
WITH ArtistGenreCount AS (
    SELECT
        ar.ArtistId,
        ar.Name AS ArtistName,
        COUNT(DISTINCT g.GenreId) AS DistinctGenreCount
    FROM Artist ar
    JOIN Album al ON ar.ArtistId = al.ArtistId
    JOIN Track t ON al.AlbumId = t.AlbumId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY ar.ArtistId, ar.Name
)
SELECT
    ArtistName,
    DistinctGenreCount
FROM ArtistGenreCount
ORDER BY DistinctGenreCount DESC
LIMIT 1;

In [ ]:
**7. Percentage of Sales per Genre:**
Calculate the percentage of total revenue generated by each genre.


In [ ]:
WITH GenreSales AS (
    SELECT
        g.Name AS GenreName,
        SUM(il.UnitPrice * il.Quantity) AS GenreRevenue
    FROM Genre g
    JOIN Track t ON g.GenreId = t.GenreId
    JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY g.Name
),
TotalSales AS (
    SELECT SUM(GenreRevenue) AS TotalRevenue FROM GenreSales
)
SELECT
    gs.GenreName,
    gs.GenreRevenue,
    (gs.GenreRevenue * 100.0 / ts.TotalRevenue) AS PercentageOfTotal
FROM GenreSales gs
CROSS JOIN TotalSales ts
ORDER BY PercentageOfTotal DESC;

In [ ]:
**8. Average Invoice Amount per Country:**
Calculate the average invoice total for each country.


In [ ]:
SELECT
    c.Country,
    AVG(i.Total) AS AverageInvoiceAmount,
    COUNT(i.InvoiceId) AS NumberOfInvoices
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
HAVING COUNT(i.InvoiceId) > 0 -- Exclude countries with no invoices (if any)
ORDER BY AverageInvoiceAmount DESC;

In [ ]:
**9. Tracks Appearing in Most Playlists:**
Find the top 10 tracks that appear in the highest number of distinct playlists.

In [ ]:
SELECT
    t.Name AS TrackName,
    ar.Name AS ArtistName,
    COUNT(DISTINCT pt.PlaylistId) AS PlaylistCount
FROM Track t
JOIN PlaylistTrack pt ON t.TrackId = pt.TrackId
JOIN Album al ON t.AlbumId = al.AlbumId
JOIN Artist ar ON al.ArtistId = ar.ArtistId
GROUP BY t.TrackId, TrackName, ArtistName -- Group by TrackId to count playlists per track
ORDER BY PlaylistCount DESC
LIMIT 10;

In [ ]:
**10. Running Total of Monthly Sales:**
Calculate the running total of sales revenue month by month.


In [ ]:
WITH MonthlySales AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS SaleMonth, -- Extract YYYY-MM
        SUM(Total) as MonthlyRevenue
    FROM Invoice
    GROUP BY SaleMonth
)
SELECT
    SaleMonth,
    MonthlyRevenue,
    SUM(MonthlyRevenue) OVER (ORDER BY SaleMonth ASC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RunningTotalRevenue
FROM MonthlySales
ORDER BY SaleMonth;

In [ ]:
**11. Customer Purchase Frequency:**
Find the average number of days between purchases for each customer who has made more than one purchase.


In [ ]:
WITH CustomerInvoiceDates AS (
    SELECT
        CustomerId,
        InvoiceDate,
        LAG(InvoiceDate, 1) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate) AS PreviousInvoiceDate
    FROM Invoice
),
DaysBetweenPurchases AS (
    SELECT
        CustomerId,
        julianday(InvoiceDate) - julianday(PreviousInvoiceDate) AS DaysDiff -- Calculate difference in days
    FROM CustomerInvoiceDates
    WHERE PreviousInvoiceDate IS NOT NULL -- Only consider rows where a previous purchase exists
)
SELECT
    c.FirstName || ' ' || c.LastName AS CustomerName,
    AVG(dbp.DaysDiff) AS AverageDaysBetweenPurchases
FROM DaysBetweenPurchases dbp
JOIN Customer c ON dbp.CustomerId = c.CustomerId
GROUP BY dbp.CustomerId, CustomerName
HAVING COUNT(dbp.DaysDiff) > 0 -- Ensure we have at least one interval
ORDER BY AverageDaysBetweenPurchases ASC;

In [ ]:
**12. Genre Popularity Trend:**
Show the total number of tracks sold per genre for each year.


In [ ]:
SELECT
    g.Name AS GenreName,
    strftime('%Y', i.InvoiceDate) AS SaleYear,
    COUNT(il.InvoiceLineId) AS TracksSold
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
JOIN Invoice i ON il.InvoiceId = i.InvoiceId
GROUP BY GenreName, SaleYear
ORDER BY GenreName, SaleYear;

In [ ]:
**13. Customers Who Bought Tracks from Multiple Artists of the Same Genre:**
Find customers who have purchased tracks from at least two different artists within the *same* genre (e.g., bought tracks from two different Rock artists).

In [ ]:
-- Question 13 Query
WITH CustomerGenreArtist AS (
    SELECT DISTINCT -- Only need unique combinations
        c.CustomerId,
        g.GenreId,
        ar.ArtistId
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    JOIN Album al ON t.AlbumId = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
)
SELECT
    cu.FirstName || ' ' || cu.LastName AS CustomerName,
    g.Name AS GenreName,
    COUNT(cga.ArtistId) AS DifferentArtistsInGenre
FROM CustomerGenreArtist cga
JOIN Customer cu ON cga.CustomerId = cu.CustomerId
JOIN Genre g ON cga.GenreId = g.GenreId
GROUP BY cga.CustomerId, CustomerName, cga.GenreId, GenreName
HAVING COUNT(cga.ArtistId) >= 2 -- The core condition
ORDER BY CustomerName, GenreName;

In [ ]:
**14. Longest Track per Genre:**
Find the longest track (in milliseconds) for each genre.


In [ ]:
WITH TrackRankedByLength AS (
    SELECT
        t.Name AS TrackName,
        g.Name AS GenreName,
        t.Milliseconds,
        RANK() OVER (PARTITION BY g.GenreId ORDER BY t.Milliseconds DESC) as LengthRank
    FROM Track t
    JOIN Genre g ON t.GenreId = g.GenreId
)
SELECT
    GenreName,
    TrackName,
    Milliseconds
FROM TrackRankedByLength
WHERE LengthRank = 1
ORDER BY GenreName;


In [ ]:
**15. Sales Contribution by Employee's Country:**
Calculate the total sales amount generated by customers grouped by the country of their supporting employee.

In [ ]:
SELECT
    e.Country AS EmployeeCountry,
    SUM(i.Total) AS TotalSales
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY e.Country
ORDER BY TotalSales DESC;

In [ ]:
**16. Media Type Usage:**
Show the total number of tracks sold for each media type.


In [ ]:
SELECT
    mt.Name AS MediaTypeName,
    COUNT(il.InvoiceLineId) AS TracksSold
FROM MediaType mt
JOIN Track t ON mt.MediaTypeId = t.MediaTypeId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY mt.MediaTypeId, MediaTypeName
ORDER BY TracksSold DESC;

In [ ]:
**17. Artists with Only One Album:**
List artists who have exactly one album in the database.

In [ ]:
SELECT
    ar.Name AS ArtistName,
    COUNT(al.AlbumId) AS AlbumCount
FROM Artist ar
LEFT JOIN Album al ON ar.ArtistId = al.ArtistId -- LEFT JOIN to include artists even if they have 0 albums (though schema likely prevents this)
GROUP BY ar.ArtistId, ArtistName
HAVING COUNT(al.AlbumId) = 1
ORDER BY ArtistName;

In [ ]:
**18. Customer Cohort Analysis (First Purchase Month):**
Group customers by the month of their first purchase and calculate the total spending for each cohort.

In [ ]:
WITH FirstPurchase AS (
    SELECT
        CustomerId,
        MIN(InvoiceDate) as FirstInvoiceDate
    FROM Invoice
    GROUP BY CustomerId
),
CustomerCohort AS (
    SELECT
        CustomerId,
        strftime('%Y-%m', FirstInvoiceDate) as CohortMonth
    FROM FirstPurchase
)
SELECT
    cc.CohortMonth,
    SUM(i.Total) AS TotalCohortSpending,
    COUNT(DISTINCT i.CustomerId) AS CohortSize
FROM CustomerCohort cc
JOIN Invoice i ON cc.CustomerId = i.CustomerId
GROUP BY cc.CohortMonth
ORDER BY cc.CohortMonth;

In [ ]:
**19. Top 3 Tracks per Playlist:**
For each playlist, list the top 3 longest tracks (in milliseconds).


In [ ]:
WITH PlaylistTrackLengthRank AS (
    SELECT
        p.Name AS PlaylistName,
        t.Name AS TrackName,
        t.Milliseconds,
        ROW_NUMBER() OVER (PARTITION BY p.PlaylistId ORDER BY t.Milliseconds DESC) as RankInPlaylist
    FROM Playlist p
    JOIN PlaylistTrack pt ON p.PlaylistId = pt.PlaylistId
    JOIN Track t ON pt.TrackId = t.TrackId
)
SELECT
    PlaylistName,
    TrackName,
    Milliseconds
FROM PlaylistTrackLengthRank
WHERE RankInPlaylist <= 3
ORDER BY PlaylistName, RankInPlaylist;

In [ ]:
**20. Percentage Change in Monthly Sales:**
Calculate the month-over-month percentage change in total sales revenue.


In [ ]:
WITH MonthlySales AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS SaleMonth,
        SUM(Total) as MonthlyRevenue
    FROM Invoice
    GROUP BY SaleMonth
),
LaggedMonthlySales AS (
    SELECT
        SaleMonth,
        MonthlyRevenue,
        LAG(MonthlyRevenue, 1, 0.0) OVER (ORDER BY SaleMonth) AS PreviousMonthRevenue -- Default to 0.0 for first month
    FROM MonthlySales
)
SELECT
    SaleMonth,
    MonthlyRevenue,
    PreviousMonthRevenue,
    CASE
        WHEN PreviousMonthRevenue = 0 OR PreviousMonthRevenue IS NULL THEN NULL -- Avoid division by zero or for the first month
        ELSE ( (MonthlyRevenue - PreviousMonthRevenue) * 100.0 / PreviousMonthRevenue )
    END AS PercentageChange
FROM LaggedMonthlySales
ORDER BY SaleMonth;